# SAT Dataset Cleaning & Database Insertion


This notebook cleans SAT score data and appends it to a PostgreSQL database.

## 1. Load and Inspect Dataset

In [59]:
import pandas as pd
import numpy as np

df = pd.read_csv('/Users/nigar_kauser/Documents/Webeet_internship/Github_repo/_onboarding_data/daily_tasks/day_4/day_4_datasets/sat-results.csv')
df.head()


,DBN,SCHOOL NAME,Num of SAT Test Takers,SAT Critical Reading Avg. Score,SAT Math Avg. Score,SAT Writing Avg. Score,SAT Critical Readng Avg. Score,internal_school_id,contact_extension,pct_students_tested,academic_tier_rating
0,01M292,HENRY STREET SCHOOL FOR INTERNATIONAL STUDIES,29,355,404,363,355,218160,x345,78%,2.0
1,01M448,UNIVERSITY NEIGHBORHOOD HIGH SCHOOL,91,383,423,366,383,268547,x234,NaN,3.0
2,01M450,EAST SIDE COMMUNITY SCHOOL,70,377,402,370,377,236446,x123,NaN,3.0
3,01M458,FORSYTH SATELLITE ACADEMY,7,414,401,359,414,427826,x123,92%,4.0
4,01M509,MARTA VALLE HIGH SCHOOL,44,390,433,384,390,672714,x123,92%,2.0


## 2. Clean and Normalize Data

In [60]:
# Normalize column names
df.columns = (
    df.columns.str.lower() # Convert to lowercase
    .str.strip() # Remove leading/trailing whitespace
    .str.replace(' ', '_') # Replace spaces with underscores
    .str.replace('.', '', regex=False) # Remove periods
)
# Drop unnecessary columns
df = df.drop(columns=[
    'sat_critical_readng_avg_score',
    'internal_school_id',
    'contact_extension',
    'academic_tier_rating'
])
# Rename columns for clarity             
df = df.rename(columns={
    'num_of_sat_test_takers': 'num_test_takers',
    'sat_critical_reading_avg_score': 'sat_reading',
    'sat_math_avg_score': 'sat_math',
    'sat_writing_avg_score': 'sat_writing'
})

#df.info()


# Convert columns to numeric, coercing errors to NaN
for col in ['num_test_takers','sat_reading','sat_math','sat_writing']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Clean pct_students_tested column to numeric: strip '%' and convert to [0, 1] range
df['pct_students_tested'] = (
    df['pct_students_tested']
    .str.rstrip('%')
    .pipe(pd.to_numeric, errors='coerce') / 100
)    

# Filter out rows with invalid SAT scores
#for col in ['sat_reading','sat_math','sat_writing']:
 #   df = df[df[col].between(200,800)]


# Replacing invalid SAT scores with NaN instead of removing rows
for col in ['sat_reading', 'sat_math', 'sat_writing']:
    df.loc[~df[col].between(200, 800), col] = np.nan    

# Remove rows with any NaN values and duplicates
#df = df.dropna().drop_duplicates()
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 493 entries, 0 to 492
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   dbn                  493 non-null    object 
 1   school_name          493 non-null    object 
 2   num_test_takers      435 non-null    float64
 3   sat_reading          435 non-null    float64
 4   sat_math             430 non-null    float64
 5   sat_writing          435 non-null    float64
 6   pct_students_tested  376 non-null    float64
dtypes: float64(5), object(2)
memory usage: 27.1+ KB


## 3. Save Cleaned CSV

In [61]:
# Save cleaned data to a new CSV file
df.to_csv('cleaned_sat_results.csv', index=False)
print('cleaned_sat_results.csv saved successfully')

cleaned_sat_results.csv saved successfully


## 4. Append Data to PostgreSQL

In [62]:
from sqlalchemy import create_engine

# --- 1. Define your connection string ---
connection_string = (
    "postgresql+psycopg2://neondb_owner:a9Am7Yy5r9_T7h4OF2GN" ### note that password changes all of the time
    "@ep-falling-glitter-a5m0j5gk-pooler.us-east-2.aws.neon.tech/neondb"
    "?sslmode=require&channel_binding=require"
)

# --- 2. Create SQLAlchemy engine ---
engine = create_engine(connection_string)

df.to_sql(
    'nigar_kauser_sat_scores',
    engine,
    if_exists='replace',
    index=False,
    method='multi'
)

print('Data appended successfully')

Data appended successfully


In [64]:
pd.read_sql(
    "SELECT * FROM nigar_kauser_sat_scores WHERE sat_reading IS NULL LIMIT 5",
    engine
)


,dbn,school_name,num_test_takers,sat_reading,sat_math,sat_writing,pct_students_tested
0,02M392,MANHATTAN BUSINESS ACADEMY,None,None,None,None,0.85
1,02M393,BUSINESS OF SPORTS SCHOOL,None,None,None,None,NaN
2,02M399,THE HIGH SCHOOL FOR LANGUAGE AND DIPLOMACY,None,None,None,None,0.85
3,02M427,MANHATTAN ACADEMY FOR ARTS & LANGUAGE,None,None,None,None,NaN
4,02M437,HUDSON HIGH SCHOOL OF LEARNING TECHNOLOGIES,None,None,None,None,NaN
